In [ ]:
import os, glob, time, subprocess, collections
import cv2, numpy as np, pandas as pd
from google.colab import drive, files
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
!pip -q install ultralytics lapx
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/adp_project'
W = f'{BASE}/runs/yolo11s_bdd/weights/best.pt'
assert os.path.exists(W), 'best.pt not found'
clips = sorted(glob.glob(f'{BASE}/videos/*.mp4'))
assert clips, 'No clips in adp_project/videos - upload them to Drive first'
print('clips found:', [os.path.basename(c) for c in clips])
OUTD = f'{BASE}/outputs/tracking'; os.makedirs(OUTD, exist_ok=True)
MAX_SEC, MAX_W = 20, 1280

summary = []
for clip in clips:
    name = os.path.splitext(os.path.basename(clip))[0]
    cap = cv2.VideoCapture(clip)
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 30
    w, h = int(cap.get(3)), int(cap.get(4)); cap.release()
    ow = min(w, MAX_W); oh = int(round(h * ow / w / 2) * 2); ow = ow - ow % 2
    limit = int(fps_in * MAX_SEC)
    raw = f'/content/{name}_raw.mp4'
    vw = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*'mp4v'), fps_in, (ow, oh))
    model = YOLO(W)
    it = iter(model.track(source=clip, tracker='bytetrack.yaml', persist=True, conf=0.25,
                          imgsz=640, stream=True, verbose=False))
    n, t_track, life, dets = 0, 0.0, collections.Counter(), 0
    while n < limit:
        t0 = time.perf_counter()
        try:
            r = next(it)
        except StopIteration:
            break
        t_track += time.perf_counter() - t0
        n += 1
        if r.boxes is not None and r.boxes.id is not None:
            ids = r.boxes.id.int().tolist()
            dets += len(ids)
            for i in ids:
                life[i] += 1
        frame = r.plot()
        if (ow, oh) != (w, h):
            frame = cv2.resize(frame, (ow, oh))
        vw.write(frame)
    vw.release()
    final = f'{OUTD}/{name}_tracked.mp4'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', raw, '-vcodec', 'libx264',
                    '-pix_fmt', 'yuv420p', final], check=True)
    lens = list(life.values())
    summary.append(dict(clip=name, input_size=f'{w}x{h}', frames=n, track_fps=round(n / t_track, 1),
                        unique_ids=len(life), median_track_len=int(np.median(lens)) if lens else 0,
                        tracks_under_5_frames=sum(1 for v in lens if v < 5),
                        dets_per_frame=round(dets / max(n, 1), 1)))
    print(summary[-1])

df = pd.DataFrame(summary); print('\n', df.to_string(index=False))
df.to_csv(f'{OUTD}/tracking_summary.csv', index=False)
files.download(f'{OUTD}/tracking_summary.csv')
print('videos saved in Drive:', sorted(glob.glob(f'{OUTD}/*.mp4')))

In [ ]:
import os, time, collections, yaml, glob
import cv2, numpy as np, pandas as pd
from google.colab import drive, files
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
!pip -q install ultralytics lapx
import ultralytics
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/adp_project'
W = f'{BASE}/runs/yolo11s_bdd/weights/best.pt'
assert os.path.exists(W), 'best.pt not found'
clips = sorted(glob.glob(f'{BASE}/videos/*.mp4')); assert clips, 'no clips'

src = os.path.join(os.path.dirname(ultralytics.__file__), 'cfg', 'trackers', 'bytetrack.yaml')
cfg = yaml.safe_load(open(src)); print('default ByteTrack config:', cfg)
tuned = dict(cfg)
if 'track_buffer' in tuned: tuned['track_buffer'] = 60
else: print('track_buffer key not found - config C will only change the image size')
yaml.safe_dump(tuned, open('/content/bytetrack_tuned.yaml', 'w'))

VEH = [0, 1, 2, 3, 4, 6, 7]   # person, rider, car, truck, bus, motor, bike
CONFIGS = {'A baseline':           ('bytetrack.yaml', 0.25, 640, None),
           'B people+vehicles':    ('bytetrack.yaml', 0.35, 640, VEH),
           'C B + 960px + buffer': ('/content/bytetrack_tuned.yaml', 0.35, 960, VEH)}

def run(clip, tracker, conf, imgsz, classes, sec=10):
    cap = cv2.VideoCapture(clip); fps = cap.get(cv2.CAP_PROP_FPS) or 30; cap.release()
    limit = int(fps * sec)
    gen = YOLO(W).track(source=clip, tracker=tracker, persist=True, conf=conf, imgsz=imgsz,
                        classes=classes, stream=True, verbose=False)
    n, life, dets, t = 0, collections.Counter(), 0, 0.0
    while n < limit:
        t0 = time.perf_counter()
        try: r = next(gen)
        except StopIteration: break
        t += time.perf_counter() - t0; n += 1
        if r.boxes is not None and r.boxes.id is not None:
            ids = r.boxes.id.int().tolist(); dets += len(ids)
            for i in ids: life[i] += 1
    gen.close()
    lens = list(life.values())
    return dict(frames=n, unique_ids=len(life), max_id=max(life) if life else 0,
                median_len=int(np.median(lens)) if lens else 0,
                pct_under_5=round(100 * sum(v < 5 for v in lens) / max(len(lens), 1)),
                dets_per_frame=round(dets / max(n, 1), 1), fps=round(n / max(t, 1e-6), 1))

rows = []
for c in clips:
    for label, (trk, conf, sz, cls) in CONFIGS.items():
        d = run(c, trk, conf, sz, cls); d.update(clip=os.path.basename(c)[:-4], config=label)
        rows.append(d); print(d)
df = pd.DataFrame(rows)[['clip', 'config', 'frames', 'unique_ids', 'max_id', 'median_len', 'pct_under_5', 'dets_per_frame', 'fps']]
print('\n', df.to_string(index=False))
print('\nAverages per config:\n', df.groupby('config')[['unique_ids', 'median_len', 'pct_under_5', 'dets_per_frame', 'fps']].mean().round(1))
os.makedirs(f'{BASE}/outputs/tracking', exist_ok=True)
df.to_csv(f'{BASE}/outputs/tracking/tracking_comparison.csv', index=False)
files.download(f'{BASE}/outputs/tracking/tracking_comparison.csv')

In [ ]:
import os, glob, subprocess, collections
import cv2, numpy as np, pandas as pd
from google.colab import drive, files
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
!pip -q install ultralytics lapx
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/adp_project'
W = f'{BASE}/runs/yolo11s_bdd/weights/best.pt'
assert os.path.exists(W), 'best.pt not found'
OUTD = f'{BASE}/outputs/tracking_final'; os.makedirs(OUTD, exist_ok=True)
VEH = [0, 1, 2, 3, 4, 6, 7]          # person, rider, car, truck, bus, motor, bike
MAX_SEC, MAX_W, CONF = 20, 1280, 0.35
rows, to_download = [], []

for name in ['clip2', 'clip4', 'clip5']:
    src = f'{BASE}/videos/{name}.mp4'
    assert os.path.exists(src), f'{src} missing'
    cap = cv2.VideoCapture(src); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w, h = int(cap.get(3)), int(cap.get(4)); cap.release()
    ow = min(w, MAX_W) // 2 * 2; oh = int(round(h * ow / w / 2) * 2)
    limit = int(fps * MAX_SEC)
    raw = f'/content/{name}_final_raw.mp4'
    vw = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*'mp4v'), fps, (ow, oh))
    gen = YOLO(W).track(source=src, tracker='bytetrack.yaml', persist=True, conf=CONF,
                        imgsz=640, classes=VEH, stream=True, verbose=False)
    n, life, mid, last = 0, collections.Counter(), None, None
    while n < limit:
        try:
            r = next(gen)
        except StopIteration:
            break
        n += 1
        if r.boxes is not None and r.boxes.id is not None:
            for i in r.boxes.id.int().tolist():
                life[i] += 1
        frame = r.plot(conf=False, line_width=2)
        if (ow, oh) != (w, h):
            frame = cv2.resize(frame, (ow, oh))
        vw.write(frame); last = frame
        if n == limit // 2:
            mid = frame.copy()
    gen.close(); vw.release()
    final = f'{OUTD}/{name}_tracked_final.mp4'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', raw, '-vcodec', 'libx264',
                    '-pix_fmt', 'yuv420p', final], check=True)
    jpg = f'/content/{name}_tracked_frame.jpg'
    cv2.imwrite(jpg, mid if mid is not None else last, [cv2.IMWRITE_JPEG_QUALITY, 85])
    lens = list(life.values())
    rows.append(dict(clip=name, config='B conf0.35 vehicles+people', seconds=round(n / fps, 1), frames=n,
                     unique_ids=len(life), median_len=int(np.median(lens)) if lens else 0,
                     pct_under_5=round(100 * sum(v < 5 for v in lens) / max(len(lens), 1))))
    print(rows[-1]); to_download.append(jpg)

df = pd.DataFrame(rows); print('\n', df.to_string(index=False))
df.to_csv('/content/tracking_final_summary.csv', index=False)
to_download.append('/content/tracking_final_summary.csv')
for f in to_download:
    files.download(f)
print('videos in Drive:', sorted(glob.glob(f'{OUTD}/*.mp4')))